# Optimized User-based Collaborative Filtering Evaluation

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix, vstack
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from sklearn.model_selection import KFold


In [2]:
def build_sparse_matrix(ratings_df):
    user_ids = ratings_df["userId"].unique()
    movie_ids = ratings_df["movieId"].unique()

    user_map = {uid: idx for idx, uid in enumerate(user_ids)}
    movie_map = {mid: idx for idx, mid in enumerate(movie_ids)}
    reverse_movie_map = {idx: mid for mid, idx in movie_map.items()}

    row = ratings_df["userId"].map(user_map)
    col = ratings_df["movieId"].map(movie_map)
    data = ratings_df["rating"]

    sparse_matrix = csr_matrix((data, (row, col)), shape=(len(user_map), len(movie_map)))
    return sparse_matrix, user_map, movie_map, reverse_movie_map

In [3]:
from sklearn.preprocessing import normalize
import faiss
import numpy as np

def build_faiss_index_batched(sparse_matrix):
    dense = normalize(sparse_matrix.toarray().astype("float32"))
    d = dense.shape[1]

    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist=10, metric=faiss.METRIC_INNER_PRODUCT)
    index.train(dense)
    index.add(dense)
    return index

In [4]:
# Load datasets
ratings_df = pd.read_csv("../data/ratings.csv")
movies_df = pd.read_csv("../data/movies.csv")


In [5]:
def create_per_user_split(ratings_df, test_ratio=0.2, min_ratings=5, seed=42):
    train_rows, test_rows = [], []

    for user_id, user_ratings in ratings_df.groupby("userId"):
        if len(user_ratings) < min_ratings:
            train_rows.append(user_ratings)
            continue

        shuffled = user_ratings.sample(frac=1.0, random_state=seed)
        split = int(len(shuffled) * test_ratio)
        test_rows.append(shuffled.iloc[:split])
        train_rows.append(shuffled.iloc[split:])

    train_df = pd.concat(train_rows)
    test_df = pd.concat(test_rows)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [6]:
def sample_users(ratings_df, min_ratings=10, sample_size=5000, seed=42):
    users = ratings_df['userId'].value_counts()
    eligible_users = users[users >= min_ratings].index
    np.random.seed(seed)
    sampled_users = np.random.choice(eligible_users, size=sample_size, replace=False)
    return ratings_df[ratings_df['userId'].isin(sampled_users)].copy()

In [7]:
def predict_rating_dynamic(user_id, movie_id, sparse_matrix, user_map, movie_map, k=50, min_overlap=3, min_neighbors=15):
    if user_id not in user_map or movie_id not in movie_map:
        return None

    user_idx = user_map[user_id]
    movie_idx = movie_map[movie_id]
    user_vector = sparse_matrix[user_idx]
    if user_vector.nnz == 0:
        return None

    user_dense = user_vector.toarray().astype("float32")[0]
    user_rated_mask = user_dense != 0
    user_rated_count = np.count_nonzero(user_rated_mask)

    column = sparse_matrix[:, movie_idx]
    rated_user_indices = column.nonzero()[0]
    if len(rated_user_indices) < min_neighbors:
        return None

    rated_user_vectors = sparse_matrix[rated_user_indices].toarray().astype("float32")
    dense_subset = normalize(rated_user_vectors)
    norm_target = normalize(user_dense.reshape(1, -1).astype("float32"))

    d = dense_subset.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(dense_subset)

    D, I = index.search(norm_target, min(k, len(rated_user_indices)))
    similarities = D[0]
    top_indices = I[0]

    weighted_scores, weights = [], []

    for sim, local_idx in zip(similarities, top_indices):
        neighbor_vector = rated_user_vectors[local_idx]
        rating = neighbor_vector[movie_idx]
        if rating == 0:
            continue

        overlap = np.sum(user_rated_mask & (neighbor_vector != 0))
        if overlap < min_overlap:
            continue

        weight = sim * (overlap / (user_rated_count + 1e-10))
        weighted_scores.append(rating * weight)
        weights.append(weight)

    if len(weights) < min_neighbors:
        return None

    return float(np.clip(np.sum(weighted_scores) / np.sum(weights), 0.5, 5.0))

In [8]:
from sklearn.metrics import mean_squared_error

def evaluate_rmse_dynamic(test_df, sparse_matrix, user_map, movie_map, k=50):
    actuals, preds = [], []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating RMSE"):
        pred = predict_rating_dynamic(
            row["userId"],
            row["movieId"],
            sparse_matrix,
            user_map,
            movie_map,
            k=k
        )
        if pred is not None:
            actuals.append(row["rating"])
            preds.append(pred)

    if not preds:
        print("⚠️ No predictions made.")
        return float('nan')

    return np.sqrt(mean_squared_error(actuals, preds))

In [9]:
# 1. Sample and split
ratings_sampled = sample_users(ratings_df, sample_size=1000)
train_df, test_df = create_per_user_split(ratings_sampled, test_ratio=0.2)

# 2. Build matrix
sparse_matrix, user_map, movie_map, reverse_movie_map = build_sparse_matrix(train_df)

# 3. Evaluate
rmse = evaluate_rmse_dynamic(test_df, sparse_matrix, user_map, movie_map, k=50)
print(f"RMSE of dynamic FAISS recommender: {rmse:.4f}")

Evaluating RMSE: 100%|██████████| 29855/29855 [02:35<00:00, 191.93it/s]

RMSE of dynamic FAISS recommender: 0.9647


In [10]:
def cross_val_user_stratified_sampled_dynamic(ratings_df, k_folds=5, sample_size=1000, k=50):
    sampled_users = sample_users_for_cv(ratings_df, sample_size=sample_size)
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    fold_rmses = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(sampled_users), 1):
        print(f"\nFold {fold}/{k_folds}")
        train_users = sampled_users[train_idx]
        test_users = sampled_users[test_idx]

        train_ratings_all = ratings_df[ratings_df["userId"].isin(train_users)].copy()
        test_ratings_all = ratings_df[ratings_df["userId"].isin(test_users)].copy()

        train_rows, test_rows = [], []
        for uid in test_users:
            user_data = test_ratings_all[test_ratings_all["userId"] == uid]
            if len(user_data) < 5:
                continue
            user_data = user_data.sample(frac=1.0, random_state=42)
            split = int(len(user_data) * 0.2)
            test_rows.append(user_data.iloc[:split])
            train_rows.append(user_data.iloc[split:])
        test_df = pd.concat(test_rows)
        test_train_df = pd.concat(train_rows)

        final_train_df = pd.concat([train_ratings_all, test_train_df])

        sparse_matrix, user_map, movie_map, reverse_movie_map = build_sparse_matrix(final_train_df)

        test_df = test_df[test_df["userId"].isin(user_map) & test_df["movieId"].isin(movie_map)]

        rmse = evaluate_rmse_dynamic(test_df, sparse_matrix, user_map, movie_map, k=k)
        print(f"  Fold RMSE: {rmse:.4f}")
        fold_rmses.append(rmse)

    print(f"\nCross-Validated RMSE (Dynamic): {np.mean(fold_rmses):.4f} ± {np.std(fold_rmses):.4f}")
    return fold_rmses

In [11]:
def sample_users_for_cv(ratings_df, min_ratings=10, sample_size=5000, seed=42):
    users = ratings_df['userId'].value_counts()
    eligible = users[users >= min_ratings].index
    np.random.seed(seed)
    sampled_users = np.random.choice(eligible, size=sample_size, replace=False)
    return sampled_users

### 1000 users

In [12]:
cross_val_user_stratified_sampled_dynamic(ratings_df, k_folds=5, sample_size=1000)


Fold 1/5


Evaluating RMSE: 100%|██████████| 4751/4751 [00:34<00:00, 137.10it/s]


  Fold RMSE: 0.9150

Fold 2/5


Evaluating RMSE: 100%|██████████| 4815/4815 [00:37<00:00, 127.16it/s]


  Fold RMSE: 1.0050

Fold 3/5


...

Evaluating RMSE: 100%|██████████| 6866/6866 [00:41<00:00, 167.02it/s]


  Fold RMSE: 0.9866

Fold 5/5


Evaluating RMSE: 100%|██████████| 6715/6715 [00:41<00:00, 163.00it/s]

  Fold RMSE: 0.9373

Cross-Validated RMSE (Dynamic): 0.9618 ± 0.0325


[np.float64(0.9149537176583727),
 np.float64(1.0050147411274573),
 np.float64(0.9650085961035558),
 np.float64(0.9866071915066718),
 np.float64(0.9373120684766255)]

### 5000 users

In [13]:
cross_val_user_stratified_sampled_dynamic(ratings_df, k_folds=5, sample_size=5000)


Fold 1/5


Evaluating RMSE: 100%|██████████| 29222/29222 [24:22<00:00, 19.98it/s] 


  Fold RMSE: 0.9534

Fold 2/5


Evaluating RMSE: 100%|██████████| 30326/30326 [24:14<00:00, 20.85it/s] 


  Fold RMSE: 0.9274

Fold 3/5


...

Evaluating RMSE: 100%|██████████| 28291/28291 [25:01<00:00, 18.85it/s] 


  Fold RMSE: 0.9386

Fold 5/5


Evaluating RMSE: 100%|██████████| 32609/32609 [27:58<00:00, 19.42it/s] 


  Fold RMSE: 0.9155

Cross-Validated RMSE (Dynamic): 0.9317 ± 0.0131


[np.float64(0.9533661726392423),
 np.float64(0.9273952986485682),
 np.float64(0.9236746669696403),
 np.float64(0.9386289149016408),
 np.float64(0.9155418742543375)]